# Step 1 : Neural Data Preprocessing

# Importing the nessasary libraries

In [1]:
# Importing the nessasary libraries
import numpy as np
import pandas as pd
import json  # for reading the .json data file
import os
from scipy.signal import butter, filtfilt

# Importing the data

In [2]:
# Importing the data
path = r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Neural Data\DBS ATN DSB Case 1\D. Siragusa\3.5.26\Time stamp 1447\Report_Json_Session_Report_20260305T152114.json"
with open(path, 'r') as f:
    data = json.load(f)  # Data is in a python dict
data['BrainSenseTimeDomain'][0]['Channel'], data['BrainSenseTimeDomain'][1]['Channel']

('ZERO_THREE_LEFT', 'ZERO_THREE_RIGHT')

## Seperating Neural Activity Data

### Only keeping important LFP data channels

In [9]:
# channel_list = [
#     'CECOG_LF_1___01___Macro_LF___01\x00',
#     'CECOG_LF_1___02___Macro_LF___02\x00',
#     'CECOG_LF_1___03___Macro_LF___03\x00',
#     'CECOG_LF_1___04___Macro_LF___04\x00',
#     'CECOG_LF_1___05___Macro_LF___05\x00',
#     'CECOG_LF_1___06___Macro_LF___06\x00',
#     'CECOG_LF_1___07___Macro_LF___07\x00',
#     'CECOG_LF_1___08___Macro_LF___08\x00',
#     'CECOG_LF_1___09___Macro_LF___09\x00',
#     'CECOG_LF_1___10___Macro_LF___10\x00',
#     'CECOG_LF_1___11___Macro_LF___11\x00',
#     'CECOG_LF_1___12___Macro_LF___12\x00',
#     'CECOG_LF_1___13___Macro_LF___13\x00',
#     'CECOG_LF_1___14___Macro_LF___14\x00',
#     'CECOG_LF_1___15___Macro_LF___15\x00',
#     'CECOG_LF_1___16___Macro_LF___16\x00'
# ]

# marker_list = [
#     'CPORT__1',
#     'CPORT__1_KHz'
# ]

In [3]:
# Creating a dataframe from BrainSenseTimeDomain channels
channel_data = {
    entry['Channel']: entry['TimeDomainData']
    for entry in data['BrainSenseTimeDomain']
}
channel_df = pd.DataFrame(channel_data)
channel_df

,ZERO_THREE_LEFT,ZERO_THREE_RIGHT
0,11.035945,-5.479715
1,-1.434673,1.789295
2,-14.898526,3.466758
3,10.925586,-5.144222
4,-25.603394,9.393797
...,...,...
51307,-0.551797,17.110130
51308,39.287964,-3.019435
51309,-2.648627,13.867033
51310,20.857937,3.578589


### Renaming the channels to match the anatomical electrode locations

In [4]:
# Label mapping - rename channels to anatomical names
label_map = {
    'ZERO_THREE_LEFT': 'LFP_Left',
    'ZERO_THREE_RIGHT': 'LFP_Right'
}

channel_df = channel_df.rename(columns=label_map)

# Verify the result
print(channel_df.columns)


Index(['LFP_Left', 'LFP_Right'], dtype='str')


In [5]:
channel_df.head()

,LFP_Left,LFP_Right
0,11.035945,-5.479715
1,-1.434673,1.789295
2,-14.898526,3.466758
3,10.925586,-5.144222
4,-25.603394,9.393797


# Preprocessing the data

## Band Pass Filtering

In [6]:
sampling_rate = data['BrainSenseTimeDomain'][0]['SampleRateInHz']  # 250 Hz
lowcut = 0.5
highcut = 100  # lowered: Nyquist at 250 Hz is 125 Hz
order = 4

In [7]:
def butter_bandpass(lowcut, highcut, fs, order=4):
    """Design a Butterworth bandpass filter"""
    nyquist = 0.5 * fs  # Nyquist frequency
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return b, a

In [8]:
def apply_filter(data, b, a):
    """Apply filter using filtfilt (zero-phase filtering)"""
    return filtfilt(b, a, data)

In [9]:
b, a = butter_bandpass(lowcut, highcut, sampling_rate, order)
filtered_df = channel_df.copy()

for column in channel_df.columns:
    filtered_df[column] = apply_filter(channel_df[column].values, b, a)

print("Bandpass filtered data:")
print(filtered_df.head())

Bandpass filtered data:
    LFP_Left  LFP_Right
0 -10.786124  17.199680
1 -29.850085  27.405465
2 -23.584027  19.738117
3 -27.327124  25.196689
4 -27.709356  21.724876


## Notch Filtering 
To remove bands of noise

In [10]:
from scipy.signal import iirnotch
def apply_notch_filters(df, sampling_rate=1375, notch_freqs=[60, 120, 180, 240, 300], Q=30):
    """
    Apply multiple notch filters to remove line noise harmonics.
    Apply this AFTER bandpass filtering.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Bandpassed channel data
    sampling_rate : float
        Sampling rate in Hz (default: 1000)
    notch_freqs : list
        Frequencies to notch out in Hz
    Q : float
        Quality factor (default: 30)
    
    Returns:
    --------
    filtered_df : pandas.DataFrame
        Notch-filtered data
    """
    
    filtered_df = df.copy()
    
    for freq in notch_freqs:
        # Skip frequencies outside the bandpass range
        if freq >= sampling_rate / 2:
            print(f"Skipping {freq} Hz (above Nyquist frequency)")
            continue
            
        b, a = iirnotch(w0=freq, Q=Q, fs=sampling_rate)
        
        for column in filtered_df.columns:
            filtered_df[column] = filtfilt(b, a, filtered_df[column].values)
        
        print(f"Applied notch filter at {freq} Hz")
    
    return filtered_df

In [11]:
# Apply notch to your already bandpassed data
channel_df = apply_notch_filters(filtered_df, 
                                sampling_rate=sampling_rate,
                                notch_freqs=[60, 120, 180, 240, 300])

Applied notch filter at 60 Hz
Applied notch filter at 120 Hz
Skipping 180 Hz (above Nyquist frequency)
Skipping 240 Hz (above Nyquist frequency)
Skipping 300 Hz (above Nyquist frequency)


In [12]:
channel_df

,LFP_Left,LFP_Right
0,-10.677531,17.060787
1,-29.470736,27.680199
2,-23.711516,19.991164
3,-27.622499,24.875867
4,-27.588476,21.404135
...,...,...
51307,10.527353,-0.904730
51308,10.689754,-3.518362
51309,4.284015,-2.166623
51310,4.418431,-3.424425


## Saving the neural data

In [13]:
# Add CAR referenced channels to channel_df
for col in channel_df.columns:
    channel_df[col] = channel_df[col].values

print(f"channel_df now has {len(channel_df.columns)} columns: {list(channel_df.columns)}")

# Create a directory to save the processed data
if not os.path.exists("Preprocessed Data"):
    os.mkdir('Preprocessed Data')

# Save combined channel data (original + CAR referenced)
channel_df.to_csv('Preprocessed Data/Cleaned Channel Data.csv', index=False)
print(f"Saved to 'Preprocessed Data/Cleaned Channel Data.csv'")
print(f"Channels: {list(channel_df.columns)}")

channel_df now has 2 columns: ['LFP_Left', 'LFP_Right']
Saved to 'Preprocessed Data/Cleaned Channel Data.csv'
Channels: ['LFP_Left', 'LFP_Right']
